# 10장 여기서 ChatGPT 까지 (실습)

교재 `docs/book/10-beyond.md` 와 함께 본다. 이 장은 개념 지도가 중심이고, 노트북은 그중 **하나(SFT)** 만 작게 시연한다:

1. base 모델은 질문에 답하지 않는다. 이어 쓴다
2. "질문 → 답" 형식 데이터 80개로 200 스텝 더 학습(SFT)하면 형식을 따른다
3. 배운 적 없는 질문에도 그럴듯하게 답한다. **환각**이 어디서 오는지

> 전체 실행 약 3분.

In [ ]:
import torch

from shllm.config import TOKENIZER_DIR, setup_cpu
from shllm.data import WORKS
from shllm.generate import generate_text
from shllm.sft import ANSWER, PROMPT, answer, build_examples, finetune
from shllm.tokenizer import BPETokenizer
from shllm.train import load_checkpoint

setup_cpu()
tok = BPETokenizer.load(TOKENIZER_DIR / "bpe-8192.json")
try:
    base, ck = load_checkpoint("small-cpu", "best.pt")
except FileNotFoundError:
    base, ck = load_checkpoint("tiny-notebook", "best.pt")
print(f"base 모델: step {ck['step']}, {base.n_params():,} 파라미터")

## 1. base 모델에게 질문하면

In [ ]:
q = PROMPT + "봄봄의 작가는 누구인가?" + ANSWER
torch.manual_seed(0)
print(repr(generate_text(base, tok, q, max_new_tokens=30, temperature=0.8, top_k=50)))

"질문: … 답: " 이라는 형식을 본 적이 없으니 그냥 소설을 이어 쓴다. 모델이 멍청해서가 아니라 **그런 데이터로 학습된 적이 없어서**다.

## 2. 지시 데이터 만들기: 코퍼스의 작품 목록으로

In [ ]:
works = [(author, title) for author, title, _slug in WORKS]
examples = build_examples(works)
print(f"작품 {len(works)}편 × 템플릿 4 = 예제 {len(examples)}개")
print(examples[0], examples[5], sep="")

## 3. SFT, 같은 모델, 형식 있는 데이터로 200 스텝

In [ ]:
import copy

sft = copy.deepcopy(base)  # base 는 그대로 두고 복사본을 학습
losses = finetune(sft, tok, examples, steps=200, lr=1e-4, batch_size=8, log_every=50)
print(f"loss {losses[0]:.2f} → {losses[-1]:.2f}")

In [ ]:
questions = [
    "봄봄의 작가는 누구인가?",  # 학습에 있던 질문 그대로
    "동백꽃을 쓴 사람은?",  # 다른 템플릿
    "이상이 쓴 작품을 하나 말해 보라.",
    "현진건의 작품 중 하나는?",  # 학습에 없던 문장
    "홍길동전의 작가는 누구인가?",  # 코퍼스에 없는 작품 → 답을 모른다
    "오늘 날씨는 어떤가?",  # 전혀 다른 질문
]
for q in questions:
    print(f"Q: {q}\nA: {answer(sft, tok, q)}\n")

**해 보기**: `TEMPLATES` 에 `("{title}의 작가를 모르면 '모름'이라고 답하라. {title}의 작가는?", "{author}")` 같은 템플릿과, 코퍼스에 없는 작품에 대해 `"모름"` 이라 답하는 예제 몇 개를 `examples` 에 직접 추가한 뒤 다시 학습해 보라. "모른다"고 말하는 것도 **예제로 가르쳐야** 한다.

앞의 셋은 맞히고(형식과 사실을 80개 예제로 배웠다), 뒤의 셋은 **모른다고 하지 않고 그럴듯한 작가 이름을 댄다**.
이것이 **환각(hallucination)** 의 원리다. 모델은 "답: " 다음에 올 확률이 높은 토큰을 낼 뿐, 아는지 모르는지를 구분하는 장치가 없다.
ChatGPT 가 "모르겠습니다" 라고 말할 수 있는 것은 그렇게 답하는 예제와 선호 학습(RLHF)이 있었기 때문이다.

## 4. 무엇이 더 필요한가 (교재 10장)

| 단계 | 이 노트북 | 실제 |
|---|---|---|
| base 학습 | 49만 토큰, 6.8M | 수조 토큰, 수십억~수천억 |
| SFT | 80 예제, 1 형식 | 수만~수백만 대화, 다양한 작업 |
| 선호 학습 (RLHF·DPO) | 없음 | 사람이 고른 답 쪽으로 다듬기, 거절·안전 |
| 도구·검색 (RAG) | 없음 | 외부 문서를 문맥에 넣어 환각 완화 |

---
**끝.** 0장의 정의로 돌아가 보라. "지금까지의 글을 보고 다음 토큰의 확률을 계산하는 함수". 10개 장에서 만든 것이 정확히 그것이고, 나머지는 데이터와 규모다.